In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.gridspec as gridspec
import pandas as pd

from os.path import join
import os
from functools import partial
import pathlib
from pathlib import Path
import shutil

from joblib import Parallel, delayed
import joblib

import gc
import subprocess

from credit.pbs import get_num_cpus

In [ ]:
# parameters
forecast_dir = "/glade/derecho/scratch/dkimpara/goes_10km_train/wx_inverted/forecasts/2022-07-01T00:25:06"
c13_only = 0
num_cpus = 2
test = 0
config = None
delay = 50
dpi = 300
max_steps = 12
channel_list = ""  # e.g. "11,13,15"; overrides c13_only when set


In [ ]:
if channel_list:
    channels = [int(c) for c in str(channel_list).split(",")]
elif c13_only:
    channels = [13]
else:
    channels = [7, 8, 9, 10, 13]


In [ ]:
# ---- format detection: old netCDF dir OR new zarr store ----
#   netCDF: <forecasts>/<init>/<valid>.nc  (one file per step) -> original code path
#   zarr:   <forecasts>/<init>.zarr        (one store per init; "<init>" is accepted too)
_p = str(forecast_dir).rstrip("/")
if os.path.isdir(_p) and not _p.endswith(".zarr"):
    fmt, forecast_path = "netcdf", _p
elif os.path.isdir(_p if _p.endswith(".zarr") else _p + ".zarr"):
    fmt, forecast_path = "zarr", (_p if _p.endswith(".zarr") else _p + ".zarr")
else:
    raise FileNotFoundError(f"no netCDF dir {_p} and no zarr store {_p}.zarr")

# init time string and the forecasts dir (where gifs/ goes), same for both formats
init_str = Path(forecast_path).name.removesuffix(".zarr")
forecasts_parent = str(Path(forecast_path).parent)

if fmt == "zarr":
    with xr.open_zarr(forecast_path) as _ds:
        _available = [int(c) for c in _ds.channel.values]
        n_zarr_steps = _ds.sizes["t"]
    if not os.path.exists(forecast_path + ".complete"):
        print(f"WARNING: {forecast_path} has no .complete marker; unwritten steps plot as NaN")
    _missing = [c for c in channels if c not in _available]
    if _missing:
        print(f"WARNING: channels {_missing} not in this zarr forecast (it has {_available}); skipping them")
    channels = [c for c in channels if c in _available]
    if not channels:
        raise ValueError(f"none of the requested channels are in this forecast (it has {_available})")

print(f"{fmt} forecast: {forecast_path}  init {init_str}  channels {channels}")


In [ ]:
def plot_forecast_and_diff(
    ds, true_ds, init_ds, forecast_dir, channels, img_idx=-1, diff_max=25
):
    true_ds["t"] = ds.t

    for channel in channels:
        fig = plt.figure(figsize=(12, 5))
        gs = gridspec.GridSpec(
            1,
            5,
            width_ratios=[1, 1, 0.1, 1, 0.1],
            wspace=0.1,
            figure=fig,
        )

        i = 0
        # Create axes
        ax1 = fig.add_subplot(gs[i, 0], projection=ccrs.PlateCarree())
        ax2 = fig.add_subplot(gs[i, 1], projection=ccrs.PlateCarree())
        cax = fig.add_subplot(gs[i, 2])  # colorbar axis (between 2nd & 3rd)
        ax3 = fig.add_subplot(gs[i, 3], projection=ccrs.PlateCarree())
        cax2 = fig.add_subplot(gs[i, 4])

        # Load the raw prediction
        pred = ds.sel(
            channel=channel
        ).BT_or_R  # .isel(latitude=slice(11,-10), longitude=slice(19,-18))

        # --- THE FIX FOR ENSEMBLES ---
        if "ensemble_member_label" in pred.dims:
            # Option A: Plot the ensemble mean (Recommended)
            pred = pred.mean(dim="ensemble_member_label")

            # Option B: If you prefer to plot just one specific member:
            # pred = pred.isel(ensemble_member_label=3)
        # -----------------------------

        # Now squeeze to remove time/channel dims, making it strictly 2D
        pred = pred.squeeze()
        true = true_ds.sel(channel=channel).BT_or_R.squeeze()
        init = init_ds.sel(channel=channel).BT_or_R.squeeze()

        if channel > 4:
            pred.plot(
                ax=ax1,
                transform=ccrs.PlateCarree(),
                cmap="Spectral_r",
                vmax=300,
                vmin=200,
                add_colorbar=False,
            )
            im2 = true.plot(
                ax=ax2,
                transform=ccrs.PlateCarree(),
                cmap="Spectral_r",
                vmax=300,
                vmin=200,
                add_colorbar=False,
            )

            diff = pred - true
            diffbar = diff.plot(
                ax=ax3,
                transform=ccrs.PlateCarree(),
                cmap="seismic",
                vmax=diff_max,
                vmin=-diff_max,
                add_colorbar=False,
            )
            ax3.set_title("Prediction - Observations")
        else:  # channel 4
            vmin, vmax = 0.0, 0.2
            pred.plot(
                ax=ax1,
                transform=ccrs.PlateCarree(),
                cmap="Spectral_r",
                vmax=vmax,
                vmin=vmin,
                add_colorbar=False,
            )
            im2 = true.plot(
                ax=ax2,
                transform=ccrs.PlateCarree(),
                cmap="Spectral_r",
                vmax=vmax,
                vmin=vmin,
                add_colorbar=False,
            )

            diff = pred - true
            diffbar = diff.plot(
                ax=ax3,
                transform=ccrs.PlateCarree(),
                cmap="seismic",
                vmax=0.1,
                vmin=-0.1,
                add_colorbar=False,
            )

        ax1.set_title("Prediction" if img_idx > 0 else "IC")
        ax2.set_title("Observations")
        ax3.set_title("Prediction - Observations")

        ax1.set_yticks(list(range(-50, 51, 25)))

        for ax in [ax1, ax2, ax3]:
            ax.add_feature(cfeature.COASTLINE)
            ax.set_xticks(list(range(-120, -29, 30)))

        for ax in [cax, cax2]:
            ax.axis("off")
            ax.get_xaxis().set_ticks([])
            ax.get_yaxis().set_ticks([])

        cbar = fig.colorbar(
            im2, ax=cax, orientation="vertical", fraction=1, shrink=0.75
        )
        cbar = fig.colorbar(
            diffbar, ax=cax2, orientation="vertical", fraction=1, shrink=0.75
        )

        # --- TITLE LOGIC WITH LEAD TIME ---
        pred_time = pd.Timestamp(pred.t.values.item())
        init_time = pd.Timestamp(init_ds.t.values.item())
        lead_time = pred_time - init_time
        
        # Convert total seconds to minutes and round to the nearest minute
        total_minutes = round(lead_time.total_seconds() / 60)
        hours, minutes = divmod(total_minutes, 60)
        
        if img_idx == 0:
            lead_str = "Init"
        else:
            lead_str = f"+{int(hours):02d}h{int(minutes):02d}m"

        fig.suptitle(
            f"Channel {channel}, forecast step {img_idx:02} ({lead_str})\n{pred_time.strftime('%Y-%m-%dT%H:%M:%S')}"
        )

        # fig.suptitle(f"Channel {channel}, forecast step {img_idx:02}\n{pd.Timestamp(pred.t.values[0]).strftime('%Y-%m-%dT%H:%M:%S')}")
        # fig.suptitle(f"Channel {channel}, forecast step {img_idx:02}\n{pd.Timestamp(pred.t.values.item()).strftime('%Y-%m-%dT%H:%M:%S')}")

        figname = f"C{channel:02}_step{img_idx:02}.png"

        save_dir = join(forecasts_parent, "gifs", f"gifs_{init_str}/C{channel:02}")

        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(join(save_dir, figname), format="png", dpi=dpi)

        plt.close(fig)

In [ ]:
num_cpus = max(1, get_num_cpus() - 1)

print(f"using {num_cpus} cpus")
print(f"processing {forecast_dir}")

if fmt == "netcdf":
    files = sorted(
        [f for f in os.listdir(forecast_path) if os.path.isfile(join(forecast_path, f))]
    )[:max_steps]
else:  # zarr: "files" are step indices into the store
    files = list(range(min(n_zarr_steps, max_steps)))

if test:
    print("WARNING: test mode!")
    files = files[:2]
    num_cpus = 2


def make_frame(forecast_dir, channels, file_and_index):
    k, file = file_and_index

    img_idx = k + 1

    zarr_ds = (
        xr.open_zarr(
            "/glade/derecho/scratch/bagherio/cloud.dir/datasets/goes_cloud_dataset/goes_10km_v2.zarr",
            consolidated=False,
            chunks=None,
        )
        .drop_duplicates(dim="t")
        .sortby("t")
        .transpose("t", "channel", "latitude", "longitude")
    )

    # get init data
    init_time = pd.Timestamp(init_str)
    init_ds = zarr_ds.sel(t=[init_time], method="nearest")

    if img_idx == 0:  # plot the init image
        plot_forecast_and_diff(
            init_ds, init_ds, init_ds, forecast_dir, channels, img_idx
        )
        gc.collect()
        return

    # plot the forecast and obs
    if fmt == "netcdf":
        file = join(forecast_dir, file)
        ds = xr.open_dataset(file)  # get prediction
    else:  # one step of the zarr store, shaped like a netCDF step file
        # Read the whole step with plain slices and load it into memory (~240 MB),
        # then select channels in numpy. zarr <= 3.1 mis-decodes sharded chunks when
        # a lazy selection drops a dimension (e.g. picking one channel) -> ValueError.
        store = xr.open_zarr(forecast_dir, chunks=None).isel(t=slice(file, file + 1)).load()
        ds = store[["BT_or_R"]]
        if "valid_time" in store:  # actual GOES scan time (t is the nominal step time)
            ds = ds.assign_coords(t=store["valid_time"].values)
    true_ds = zarr_ds.sel(t=ds.t, method="nearest")

    plot_forecast_and_diff(ds, true_ds, init_ds, forecast_dir, channels, img_idx)

    gc.collect()


def make_gif(forecast_dir, channels, file_and_index):
    """Run one frame; return (frame index, error text or None) so a failing frame
    is reported in the log instead of silently stopping the run."""
    import traceback
    try:
        make_frame(forecast_dir, channels, file_and_index)
        return file_and_index[0] + 1, None
    except Exception:
        return file_and_index[0] + 1, traceback.format_exc()


f = partial(make_gif, forecast_path, channels)

enum_files = [(-1, None)] + list(enumerate(files))
print(files)

result = Parallel(n_jobs=num_cpus)(
    delayed(f)(file_and_index) for file_and_index in enum_files
)

failed = [(i, err) for i, err in result if err]
print(f"frames written: {len(result) - len(failed)} of {len(result)}")
for i, err in failed[:3]:
    print(f"\n--- frame {i} FAILED ---\n{err}")
if failed and len(failed) >= len(result) - 1:
    raise RuntimeError(f"{len(failed)} of {len(result)} frames failed; first traceback printed above")


In [ ]:
gif_dir = join(forecasts_parent, "gifs", f"gifs_{init_str}")


for channel in channels:
    res = subprocess.run(
        f"magick -delay {delay} -loop 0 {join(gif_dir, f'C{channel:02}/*.png')} {join(gif_dir, f'C{channel:02}.gif')}",
        shell=True,
        capture_output=True,
        encoding="utf-8",
    )
    if res.returncode != 0 or not os.path.exists(join(gif_dir, f"C{channel:02}.gif")):
        print(f"FAILED to create gif for C{channel:02} (exit {res.returncode}); PNGs kept\n{res.stderr.strip()[-2000:]}")
        continue
    print(f"created gif for C{channel:02}")
    shutil.rmtree(join(gif_dir, f"C{channel:02}"))
